In [23]:
from neo4j import GraphDatabase

URI = "neo4j+ssc://96b5fb72.databases.neo4j.io"
AUTH = ("neo4j", "JZGlgN8450TFoV4QBU9r5wKc9M1CFJHAfKL7fTVjE1Y") 

driver = GraphDatabase.driver(URI, auth=AUTH)

In [24]:
with driver.session(database="neo4j") as session:
    print(session.run("RETURN 1").single()[0])

1


In [25]:
import pandas as pd

df = pd.read_csv(r"C:\Users\shett\Downloads\fortune cleaned.csv")
print(df.head())

   Rank             Company Ticker       Sector  \
0     1             Walmart    WMT    Retailing   
1     2              Amazon   AMZN    Retailing   
2     3               Apple   AAPL   Technology   
3     4  Unitedhealth Group    UNH  Health Care   
4     5  Berkshire Hathaway   BRKA   Financials   

                                   Industry  Profitable  Founder_is_CEO  \
0                     General Merchandisers         1.0             0.0   
1           Internet Services And Retailing         1.0             0.0   
2               Computers, Office Equipment         1.0             0.0   
3   Health Care: Insurance And Managed Care         1.0             0.0   
4  Insurance: Property And Casualty (Stock)         1.0             0.0   

   FemaleCEO  Growth_in_Jobs  Change_in_Rank  ...  \
0        0.0             0.0             0.0  ...   
1        0.0             0.0             0.0  ...   
2        0.0             0.0             1.0  ...   
3        0.0             0.0  

In [26]:
graph_data = []

for _, row in df.iterrows():
    data = {
        "Company": str(row["Company"]),
        "CEO": str(row["CEO"]),
        "Sector": str(row["Sector"]),
        "City": str(row["HeadquartersCity"]),
        "State": str(row["HeadquartersState"])
    }
    graph_data.append(data)

print(graph_data[:2])

[{'Company': 'Walmart', 'CEO': 'C. Douglas Mcmillon', 'Sector': 'Retailing', 'City': 'Bentonville', 'State': 'Arkansas'}, {'Company': 'Amazon', 'CEO': 'Andrew R. Jassy', 'Sector': 'Retailing', 'City': 'Seattle', 'State': 'Washington'}]


In [30]:
def create_graph(tx, data):
    query = """
    MERGE (c:Company {name:$Company})
    MERGE (ceo:CEO {name:$CEO})
    MERGE (s:Sector {name:$Sector})
    MERGE (ci:City {name:$City})
    MERGE (st:State {name:$State})

    MERGE (c)-[:HAS_CEO]->(ceo)
    MERGE (c)-[:BELONGS_TO]->(s)
    MERGE (c)-[:LOCATED_IN]->(ci)
    MERGE (ci)-[:IN_STATE]->(st)
    """
    tx.run(query, **data)

In [34]:
with driver.session(database="neo4j") as session:
    for data in graph_data:
        session.execute_write(create_graph, data)

print("Graph Created Successfully")

Graph Created Successfully


In [29]:
with driver.session(database="neo4j") as session:
    result = session.run("""
    MATCH (c:Company)-[:HAS_CEO]->(ceo)
    RETURN c.name, ceo.name LIMIT 5
    """)

    for record in result:
        print(record)

<Record c.name='Walmart' ceo.name='C. Douglas Mcmillon'>
<Record c.name='Amazon' ceo.name='Andrew R. Jassy'>
<Record c.name='Apple' ceo.name='Timothy D. Cook'>
<Record c.name='Unitedhealth Group' ceo.name='Andrew P. Witty'>
<Record c.name='Berkshire Hathaway' ceo.name='Warren E. Buffett'>
